In [ ]:
import pandas as pd
import numpy as np
import glob
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from scipy import stats

%cd -q 'data'


In [ ]:
def calc_kd(z,par):
    z = np.abs(z)
    # remove nans and negative par values (bad data)
    z = z[(par>=0) & ~np.isnan(par)]
    par = par[(par>=0) & ~np.isnan(par)]

    # calculate Kd from the log-linear slope of the light decay
    lnpar = np.log(par)
    
    fit = stats.linregress(x=z,y=lnpar)
    kd = np.abs(fit.slope)
    rval = np.abs(fit.rvalue)
    stderr = fit.stderr
    return kd, rval, stderr


def ooi_run_kdcalc(df):
    
    # get the start time of the file
    t = df.loc[0,'time'] 
    # check if there is no data shallower than 20 m:
    df = df[abs(df['z'])<20]
    if len(df) < 4:
        kd = np.nan
        rval = np.nan
        stderr = np.nan
        return kd, rval, stderr, t
        
    # if there is data, check for multiple casts:
    df.reset_index(drop=True, inplace=True)
    dt = df['time'].diff()
    df['cast'] = 0
    cast = 0
    for jj in range(len(df)):
        if dt[jj]>timedelta(minutes=10):
            cast += 1
            df.loc[jj,'cast'] = cast
        else:
            df.loc[jj,'cast'] = cast
    tot_casts = len(np.unique(df['cast']))
    
    # if there's only 1 cast, calculate kd once
    # otherwise loop through casts and calc for each
    if tot_casts==1:
        # pull out depth and par
        z = df['z'].to_numpy()
        par = df['par'].to_numpy()
        t = df.loc[0,'time']
        kd, rval, stderr = calc_kd(z,par)
    else:
        kd = np.empty(tot_casts)
        rval = np.empty(tot_casts)
        stderr = np.empty(tot_casts)
        t = df.loc[:tot_casts,'time']
        for jj in range(tot_casts):
            z = df.loc[df['cast']==jj,'z'].to_numpy()
            par = df.loc[df['cast']==jj,'par'].to_numpy()
            t[jj] = min(df.loc[df['cast']==jj,'time'])
            kd[jj], rval[jj], stderr[jj] = calc_kd(z,par)
            
    return kd, rval, stderr, t





# much simpler for ws dataset - only 1 cast per file
def ws_kd_calc(df):
    # pull out depth and par
    z = np.abs(df['z'].to_numpy())
    par = df['par'].to_numpy()
  
    # remove nans and negative par values (bad data)
    z = z[(par>=0) & ~np.isnan(par)]
    par = par[(par>=0) & ~np.isnan(par)]

    # only calculate over upper 10m
    par = par[z<20]
    z = z[z<20]
    
    if len(z) == 0:
        kd = np.nan
        rval = np.nan
        stderr = np.nan
    else:
        # calc ln of par 
        lnpar = np.log(par)
        # calculate Kd from the log-linear slope of the light decay
        fit = stats.linregress(x=z,y=lnpar)
#         par_pred = fit.slope*z + lnpar[0]
        kd = np.abs(fit.slope)
        rval = fit.rvalue
        stderr=fit.stderr

    return kd,rval,stderr

In [ ]:
FF = glob.glob('icesat2/ooi/data*.csv')
FF.sort()

ooi_kd_array = pd.DataFrame(data=None,
                            index=range(len(FF)),
                            columns=['time','lat','lon','kd','rval','stderr','stn']
                           )
jj = 43
    
df = pd.read_csv(FF[jj],
                 index_col=0,
                 header=0,
                 usecols=[i for i in range(6)],
                 names=['', 'time', 'lat', 'lon', 'z', 'par']
                ) #
df['time'] = pd.to_datetime(df["time"]).round('s')
df = df[abs(df['z'])<20]
df.reset_index(drop=True, inplace=True)
dt = df['time'].diff()

df['cast'] = 0
cast = 0
for jj in range(len(df)):
    if dt[jj]>timedelta(minutes=10):
        cast += 1
        df.loc[jj,'cast'] = cast
    else:
        df.loc[jj,'cast'] = cast
tot_casts = len(np.unique(df['cast']))


for jj in range(tot_casts):
    # pull out depth and par
    z = np.abs(df.loc[df['cast']==jj,'z'].to_numpy())
    par = df.loc[df['cast']==jj,'par'].to_numpy()
    
#     # sort by depth
#     par = par[np.argsort(z)]
#     z = np.sort(z)

    # remove nans and negative par values (bad data)
    z = z[(par>=0) & ~np.isnan(par)]
    par = par[(par>=0) & ~np.isnan(par)]
    
    lnpar = np.log(par)
    # calculate Kd from the log-linear slope of the light decay
    fit = stats.linregress(x=z,y=lnpar)
    kd = np.abs(fit.slope)
    rval = np.abs(fit.rvalue)
    stderr = fit.stderr
    print(stderr,rval)

#     plt.figure()
#     plt.plot(z,lnpar,'k-')
#     par_pred = fit.slope*z +fit.intercept
#     plt.plot(z,par_pred,'r-')



In [ ]:
# calculate Kd from OOI data


FF = glob.glob('icesat2/ooi/data*.csv')
FF.sort()

ooi_kd_array = pd.DataFrame(data=None,
                            index=range(len(FF)*2),
                            columns=['time','lat','lon','kd','rval','stderr']
                           )
cc = 0
for fname in FF:
    # load the file as a pandas array
    df = pd.read_csv(fname,
                     index_col=0,
                     header=0,
                     usecols=[i for i in range(6)],
                     names=['', 'time', 'lat', 'lon', 'z', 'par']
                    ) #
    df['time'] = pd.to_datetime(df["time"]).round('s')
    # ooi_kd_array.loc[jj,'stn'] = FF[jj][9:-4] # save the file name if its useful

    # calculate kd (there may be more than 1 value if there are multiple casts in 1 file)
    try:
        kd,rval,stderr,t = ooi_run_kdcalc(df)
    except:
        kd = np.nan
        rval = np.nan
        stderr = np.nan
    
    # either put in the 1 value or put in each value...super dumb way to do this but Im in a rush
    if type(kd) != np.ndarray:
        ooi_kd_array.loc[cc,'time'] = t
        ooi_kd_array.loc[cc,'lat'] = df.loc[1,'lat']
        ooi_kd_array.loc[cc,'lon'] = df.loc[1,'lon']
        ooi_kd_array.loc[cc,'kd'] = kd
        ooi_kd_array.loc[cc,'rval'] = rval
        ooi_kd_array.loc[cc,'stderr'] = stderr
        cc += 1 # raise counter
    else: 
        for jj in range(len(kd)):
            ooi_kd_array.loc[cc,'time'] = t[jj]
            ooi_kd_array.loc[cc,'lat'] = df.loc[1,'lat']
            ooi_kd_array.loc[cc,'lon'] = df.loc[1,'lon']
            ooi_kd_array.loc[cc,'kd'] = kd[jj]
            ooi_kd_array.loc[cc,'rval'] = rval[jj]
            ooi_kd_array.loc[cc,'stderr'] = stderr[jj]
            cc += 1 # raise counter

ooi_kd_array = ooi_kd_array.dropna(how='all')

# pd.DataFrame()
# d = {'time' = ff[0,1],'kd': m, 'rval': rval, 'stderr': stderr}
# df = pd.DataFrame(data=d)

ooi_kd_array.to_csv('../output/icesat2/ooi_kdcalculations10m.csv')
ooi_kd_array

In [ ]:
# calculate kd for all of gulf of mexico ctd profiles 
FF = glob.glob('icesat2/ctd_ws/data*.csv')
FF.sort()

ws_kd_array = pd.DataFrame(data=None,
                            index=range(len(FF)),
                            columns=['time','lat','lon','kd','rval','stderr']
                           )

for jj in range(len(FF)):
    df = pd.read_csv(FF[jj],
                     index_col=0,
                     header=0,
                     usecols=[i for i in [0,2,3,4,5,6,28]],
                     names=['','prof', 'time', 'lat', 'lon', 'z', 'par']
                    )
#     ws_kd_array.loc[jj,'stn'] = FF[jj][12:-4]
    
    try:
        kd,rval,stderr = ws_kd_calc(df)
    except:
        kd = np.nan
        rval = np.nan
        stderr = np.nan

    time = df.loc[1,'time']
    lat = df.loc[1,'lat']
    lon = df.loc[1,'lon']

    ws_kd_array.loc[jj,'time'] = time
    ws_kd_array.loc[jj,'lat'] = lat
    ws_kd_array.loc[jj,'lon'] = lon
    ws_kd_array.loc[jj,'kd'] = kd
    ws_kd_array.loc[jj,'rval'] = rval
    ws_kd_array.loc[jj,'stderr'] = stderr



pname = '../output/icesat2/'
ws_kd_array.to_csv(pname+'ws_kdcalculations10m.csv')
ws_kd_array

In [ ]:
# Process BCODMO data : only keep unique entries to search ICESat-2
FF = glob.glob('bcodmo/*.csv')
FF.sort()

for jj in range(len(FF)):
    df = pd.read_csv(FF[jj],header=0)
    df.drop(0,inplace=True)
    df = df.drop_duplicates(subset=['time'])
    df.reset_index(drop=True, inplace=True)

    print(FF[jj])
    print(df.head())
    
    fname = FF[jj][:-4]+'_unique.pkl'
    df.to_pickle(fname)



In [ ]:
# Process BCODMO data
FF = glob.glob('bcodmo/*.csv')
FF.sort()

# ws_kd_array = pd.DataFrame(data=None,
#                             index=range(len(FF)),
#                             columns=['time','lat','lon','kd','rval','stderr','stn']
#                            )
jj = 1

df = pd.read_csv(FF[jj],
                 header=0)
df = df.drop_duplicates(subset=['time'])
df.head()